# I3 — Stable Diffusion before / after demo (Google Colab)

Runs the whole demo on a free cloud GPU, so your laptop does no heavy work.

1. **Runtime → Change runtime type → T4 GPU → Save**
2. **Runtime → Run all** (or run the cells one by one)
3. Total time on a T4: about 10–15 minutes (most of it is the one-time ~4.5 GB model download)

Results appear at the bottom and are downloaded as a zip.

In [ ]:
# 1. Check that a GPU is attached
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "NO GPU: Runtime > Change runtime type > T4 GPU"

In [ ]:
# 2. Get the code from GitHub and install it
REPO = "https://github.com/Yash-Raj11/i3-stego-immunization.git"
%cd /content
!rm -rf i3 && git clone -q --depth 1 $REPO i3
%cd i3
!pip install -q -e ".[sd]"
!python scripts/download_models.py
!i3 keygen   # fresh demo keys for this Colab session (your real keys stay on your laptop)

## 3. Choose the photo
Run the next cell as is to use the sample photo, or set `USE_MY_PHOTO = True` to upload your own.
Only use photos you have permission to use.

In [ ]:
USE_MY_PHOTO = False
PHOTO = "assets/sample_astronaut.png"
if USE_MY_PHOTO:
    from google.colab import files
    up = files.upload()
    PHOTO = list(up.keys())[0]
print("photo:", PHOTO)

## 4. Run the demo
Settings you can change:
- `REFINE_STEPS`: steps against the real Stable Diffusion encoder (40 is a good GPU default; more = stronger, slower)
- `FACE_PSNR`: visibility limit on the face (32 default; lower = stronger protection but more visible)
- `MODE`: `untargeted` or `targeted` (try both)

In [ ]:
REFINE_STEPS = 40
FACE_PSNR = 32
MODE = "untargeted"
OUT = "reports/colab_demo"
!i3 demo "$PHOTO" --owner demo@example.com --config configs/sd_demo.yaml --out $OUT \
    --refine-steps $REFINE_STEPS --face-psnr $FACE_PSNR --mode $MODE

In [ ]:
# 5. Show the results
from IPython.display import Image, Markdown, display
display(Markdown(open(f"{OUT}/report.md").read()))
display(Image(f"{OUT}/before_after.png"))
display(Image(f"{OUT}/grid.png"))

In [ ]:
# 6. Download everything (figures, edits, report, protected image)
import shutil
from google.colab import files
shutil.make_archive("i3_results", "zip", OUT)
files.download("i3_results.zip")

### Tips
- Re-run cell 4 with different settings and a different `OUT` folder to compare (e.g. `FACE_PSNR = 30`).
- Colab sessions reset after inactivity: download results before you leave.
- Put the numbers from `report.md` into `docs/EXPERIMENTS.md` in your repo.